In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as seaborn
from collections import Counter
from tensorflow.keras.models import load_model

2026-02-26 19:11:21.637952: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-26 19:11:21.637985: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-26 19:11:21.645130: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-26 19:11:23.768795: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
import tensorflow as tf

print("Версия TensorFlow:", tf.__version__)
print("Доступные GPU:", tf.config.list_physical_devices('GPU'))
print("Устройство для вычислений:", tf.test.is_gpu_available())

Версия TensorFlow: 2.14.0
Доступные GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Устройство для вычислений: True


2026-02-26 19:11:47.868148: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-26 19:11:47.869479: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-26 19:11:47.870643: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [4]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("tongpython/cat-and-dog")

# print("Path to dataset files:", path)

In [5]:
from glob import glob

cat_files = glob(r'/home/nikita/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1/training_set/training_set/cats/*.jpg')
dog_files = glob(r'/home/nikita/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1/training_set/training_set/dogs/*.jpg')
cat_files_test = glob(r'/home/nikita/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1/test_set/test_set/cats/*.jpg')
dog_files_test = glob(r'/home/nikita/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1/test_set/test_set/dogs/*.jpg')

In [6]:
from PIL import Image

x_train = []
y_train = []
x_test = []
y_test = []

size = (200, 200)

for i in range(int(len(cat_files))):
    if i % 2 == 0:
        img = Image.open(cat_files[i])
        res = img.resize(size)
        res = np.array(res)
        x_train.append(res/255)
        y_train.append(0)
    else:
        img = Image.open(dog_files[i])
        res = img.resize(size)
        res = np.array(res)
        x_train.append(res/255)  
        y_train.append(1)

for i in range(int(len(cat_files_test))):
    if i % 2 == 0:
        img = Image.open(cat_files_test[i])
        res = img.resize(size)
        res = np.array(res)
        x_test.append(res/255)
        y_test.append(0)
    else:
        img = Image.open(dog_files_test[i])
        res = img.resize(size)
        res = np.array(res)
        x_test.append(res/255)
        y_test.append(1)

In [7]:
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

In [8]:
import keras
from tensorflow.keras.callbacks import EarlyStopping
from keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras import layers

In [9]:
# Включаем постепенный рост памяти (не захватываем всю сразу)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Режим Memory Growth включен")
    except RuntimeError as e:
        print(e)

Режим Memory Growth включен


In [11]:
model = keras.Sequential([
    keras.Input(shape = (200, 200, 3)),
    keras.layers.Conv2D(10, (5, 5), padding = 'same', strides = 2),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(10, (5, 5), padding = 'same', activation ='relu', strides = 2),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation ='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(32, activation ='relu'),
    keras.layers.Dense(2, activation ='softmax')
])

model.compile(  optimizer = keras.optimizers.Adam(learning_rate = 1e-4),
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

early_stopping = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    patience = 50,
                    min_delta = 0.01,
                    verbose = 0,
                    restore_best_weights = True
                    )

In [12]:
# watch -n 0.5 nvidia-smi
# nvitop

In [13]:
model.fit(
        x = x_train,
        y = y_train,
        batch_size = 32,
        epochs = 2000,
        verbose = 1,
        callbacks = [early_stopping], 
        shuffle = False,
        validation_split = 0.1,
        validation_data = None,
        validation_batch_size = 32
        )

2026-02-26 19:13:18.290686: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1728000000 exceeds 10% of free system memory.
2026-02-26 19:13:19.999456: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1728000000 exceeds 10% of free system memory.


Epoch 1/2000


2026-02-26 19:13:22.662929: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2026-02-26 19:13:33.528085: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7ce31ee2d6a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-26 19:13:33.528104: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 SUPER, Compute Capability 8.9
2026-02-26 19:13:33.627667: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-26 19:13:34.036846: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


113/113 [==============================] - 19s 14ms/step - loss: 0.8138 - accuracy: 0.5158 - val_loss: 0.7037 - val_accuracy: 0.5525
Epoch 2/2000
113/113 [==============================] - 1s 7ms/step - loss: 0.7346 - accuracy: 0.5561 - val_loss: 0.6802 - val_accuracy: 0.5575
Epoch 3/2000
113/113 [==============================] - 1s 6ms/step - loss: 0.6799 - accuracy: 0.5914 - val_loss: 0.6602 - val_accuracy: 0.6050
Epoch 4/2000
113/113 [==============================] - 1s 6ms/step - loss: 0.6463 - accuracy: 0.6242 - val_loss: 0.6791 - val_accuracy: 0.5800
Epoch 5/2000
113/113 [==============================] - 1s 7ms/step - loss: 0.6238 - accuracy: 0.6514 - val_loss: 0.6683 - val_accuracy: 0.6025
Epoch 6/2000
113/113 [==============================] - 1s 7ms/step - loss: 0.6134 - accuracy: 0.6594 - val_loss: 0.6611 - val_accuracy: 0.6000
Epoch 7/2000
113/113 [==============================] - 1s 7ms/step - loss: 0.5993 - accuracy: 0.6647 - val_loss: 0.6993 - val_accuracy: 0.5850
Epo

In [14]:
train_loss, keras_train_acc = model.evaluate(x_train, y_train)
test_loss, keras_test_acc = model.evaluate(x_test, y_test)
print('\nдля нейронной сети точность для обучающей \ тестовой выборки:', keras_train_acc, ' \ ', keras_test_acc)

2026-02-26 19:14:29.851195: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1920000000 exceeds 10% of free system memory.
2026-02-26 19:14:31.498151: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1920000000 exceeds 10% of free system memory.


32/32 [==============================] - 0s 5ms/step - loss: 0.6063 - accuracy: 0.6795

для нейронной сети точность для обучающей \ тестовой выборки: 0.7565000057220459  \  0.6795251965522766


In [15]:
1+1

2